In [1]:

!pip install -q vllm transformers==4.47.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.1/264.1 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/

In [2]:
!nohup vllm serve "Qwen/Qwen2.5-7B-Instruct-AWQ" \
    --quantization awq \
    --dtype half \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8001 \
    --served-model-name qwen25 \
    > vllm.log 2>&1 &

In [3]:
import time
time.sleep(90)  # هنستنى 90 ثانية عشان الموديل يحمل في الرامات
!tail -n 40 vllm.log
!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen25","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'

    device_config = DeviceConfig(device=self.device)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/vllm/config.py", line 1553, in __init__
    raise RuntimeError("Failed to infer device type")
RuntimeError: Failed to infer device type
INFO 08-12 20:48:04 __init__.py:187] No platform detected, vLLM is running on UnspecifiedPlatform
ERROR 08-12 20:48:05 engine.py:387] Failed to infer device type
ERROR 08-12 20:48:05 engine.py:387] Traceback (most recent call last):
ERROR 08-12 20:48:05 engine.py:387]   File "/usr/local/lib/python3.12/dist-packages/vllm/engine/multiprocessing/engine.py", line 378, in run_mp_engine
ERROR 08-12 20:48:05 engine.py:387]     engine = MQLLMEngine.from_engine_args(engine_args=engine_args,
ERROR 08-12 20:48:05 engine.py:387]              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ERROR 08-12 20:48:05 engine.py:387]   File "/usr/local/lib/python3.12/dist-packages/vllm/engine/multiprocessing/engine.p

In [4]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8001 > cloudflared.log 2>&1 &
import time
time.sleep(8)
!grep -o 'https://[a-zA-Z0-9-]*\.trycloudflare\.com' cloudflared.log | head -1

chmod: cannot access 'cloudflared-linux-amd64': No such file or directory


In [5]:
!ps aux | grep vllm
!echo "---"
!nvidia-smi

root        4662  0.0  0.0   7372  3544 ?        S    20:49   0:00 /bin/bash -c ps aux | grep vllm
root        4664  0.0  0.0   6480  2304 ?        S    20:49   0:00 grep vllm
---
/bin/bash: line 1: nvidia-smi: command not found


In [8]:
!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen25","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'